In [ ]:
!pip install datasets

In [ ]:
import re
from collections import Counter

import matplotlib.pyplot as plt
from datasets import load_dataset

LETTERS = 'а-щьюяіїєґ'
APOSTROPHE = chr(39)
TARGET = 300000

bad_chars = re.compile('[^' + LETTERS + APOSTROPHE + ' -]')
edge = re.compile('(?<![' + LETTERS + '])[' + APOSTROPHE + '-]|[' + APOSTROPHE + '-](?![' + LETTERS + '])')

In [ ]:
dataset = load_dataset('wikimedia/wikipedia', '20231101.uk', split='train', streaming=True)

for article in dataset:
    print(article['title'])
    print(article['text'][:300])
    break

In [ ]:
def split_sentences(text):
    text = text.replace(chr(10), ' . ')
    return re.split('(?<=[.!?]) +', text)


def clean_sentence(sentence):
    sentence = sentence.lower()
    sentence = sentence.replace('’', APOSTROPHE)
    sentence = sentence.replace('ʼ', APOSTROPHE)
    sentence = sentence.replace('`', APOSTROPHE)
    sentence = bad_chars.sub(' ', sentence)
    sentence = edge.sub(' ', sentence)

    words = []
    for word in sentence.split():
        if len(word) >= 3:
            words.append(word)
    return words

In [ ]:
print(clean_sentence('Київ — столиця України, 2023 року тут жив Іван-Петро і мав м\'яч!'))
print(clean_sentence('Це дуже коротке речення.'))

In [ ]:
sentences = []
articles_count = 0

for article in dataset:
    articles_count += 1

    for sentence in split_sentences(article['text']):
        words = clean_sentence(sentence)

        if len(words) >= 5:
            sentences.append(words)

        if len(sentences) >= TARGET:
            break

    if len(sentences) >= TARGET:
        break

    if articles_count % 500 == 0:
        print(articles_count, len(sentences))

print('статей оброблено:', articles_count)
print('речень у корпусі:', len(sentences))

In [ ]:
for i in range(5):
    print(i + 1, sentences[i])
    print()

In [ ]:
counter = Counter()
for sentence in sentences:
    counter.update(sentence)

total_words = sum(counter.values())

print('речень:', len(sentences))
print('слів усього:', total_words)
print('унікальних слів:', len(counter))

In [ ]:
top20 = counter.most_common(20)

for word, count in top20:
    print(word, count)

In [ ]:
words = [item[0] for item in top20]
counts = [item[1] for item in top20]

plt.figure(figsize=(12, 5))
plt.bar(words, counts)
plt.xticks(rotation=60)
plt.xlabel('Слово')
plt.ylabel('Кількість')
plt.title('20 найчастіших слів')
plt.tight_layout()
plt.show()